# Stage 1: inventory

Turns the downloads into a fixed, fully described list of photos that every later stage builds on, and
settles whether photos are tiled at native resolution or rescaled first (handbook Stage 1).

All logic lives in `src/inventory.py`, `src/leaf_width.py` and `src/inventory_report.py`; this notebook
only calls them. Outputs: `data/inventory.csv`, `data/paddy_inventory.csv`, `data/leaf_width_selection.csv`
and `docs/reports/inventory_report.md`.

In [ ]:
from IPython.display import Markdown, display

from src.config import REPO_ROOT, load_config
from src.inventory import dataset_inventory
from src.inventory_report import build_report, write_report
from src.leaf_width import decide, freeze_selection

config = load_config("configs/inventory.yaml")

## 1. Inventory of both datasets

Every file is listed (not only images), with checksum, perceptual hash, size, camera, capture time and JPEG quality. About a minute on 16 CPU threads.

In [ ]:
for name, relative in config["inventory"]["files"].items():
    frame = dataset_inventory(config, name)
    frame.to_csv(REPO_ROOT / relative, index=False)
    print(f"{name}: {len(frame):,} files, {int(frame['opened_ok'].sum()):,} decode fully -> {relative}")

## 2. Photos for the pixel-scale check

The 30 photos are chosen once (one per equal-width band of photo size) and frozen, so every measurer sees the same set.

In [ ]:
selected = freeze_selection(config)  # reads the frozen file if it already exists
selected

## 3. Report

In [ ]:
display(Markdown(write_report(config)))

## 4. Leaf-width measurement (people, in a terminal)

Two team members each run, on their own and without looking at the other's numbers:

```
uv run python -m src.leaf_width measure --measurer <name>
```

Then re-run this notebook (or just the cells below): `decide` applies the pre-declared rule, writes
`data/pixel_scale_decision.json`, and section 8 of the report fills in.

In [ ]:
result = decide(config)
{k: result[k] for k in ("complete", "answered")} if not result["complete"] else \
    {k: result[k] for k in ("spearman_rho", "p_value", "n_used", "n_unmeasurable", "rescale", "median_leaf_width_px")}